In [ ]:
# [Cell 1] Install the required library for parsing logs
!pip install drain3

In [ ]:
# [UPDATED CELL 2: Mount Google Drive and Point to Folder]

import os


# --- 2. !! IMPORTANT !! Define the path to your folder ---
# Change 'HDFS_dataset' to the name of the folder you uploaded to your Drive.
# This path is CASE-SENSITIVE.
base_drive_path = "/kaggle/input/datasets/platform934/hdfs-log-dataset/" # <-- EDIT THIS

# --- 3. Define the final file paths ---
# These paths will be used by all the following cells.
# These paths are based on your screenshots (HDFS and preprocessed/anomaly_label)
log_file = os.path.join(base_drive_path, "HDFS.log")
label_file = os.path.join(base_drive_path, "preprocessed/anomaly_label.csv")

# --- 4. Verify the files exist ---
print("\n--- Verifying file paths ---")
files_exist = False # Flag for next cells

if not os.path.exists(log_file):
    print(f"--- ERROR! ---")
    print(f"Log file not found at: {log_file}")
    print("Please check your 'base_drive_path' and folder structure.")
elif not os.path.exists(label_file):
    print(f"--- ERROR! ---")
    print(f"Label file not found at: {label_file}")
    print("Please check your 'base_drive_path' and folder structure.")
else:
    print("--- Files found! ---")
    print(f"Log file: {log_file}")
    print(f"Label file: {label_file}")
    print("You can now run the rest of the notebook cells.")
    files_exist = True

In [ ]:
# [Cell 3] Configure Drain3 Parser
import json
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

if files_exist:
    config = TemplateMinerConfig()
    config.sim_th = 0.5
    config.depth = 4
    config.masking_rules = [
        {"regex_pattern": r"blk_(-?\d+)", "mask_with": "BLOCK_ID"},
        {"regex_pattern": r"(/|)([0-9]+\.){3}[0-9]+(:[0-9]+)?", "mask_with": "IP"},
        {"regex_pattern": r"[0-9]+", "mask_with": "NUM"}
    ]
    parser = TemplateMiner(config=config)
    print("Drain3 parser configured.")

In [ ]:
# [Cell 4] Define Parsing Function
import pandas as pd
import re
import time

if files_exist:
    # Regex to find Block IDs (Session IDs)
    block_id_regex = re.compile(r"(blk_(-?\d+))")

    def parse_hdfs_log(logfile_path, label_file_path, parser):
        print(f"Loading labels from {label_file_path}...")
        try:
            label_df = pd.read_csv(label_file_path)
            # Create a dictionary for fast lookup: BlockId -> Label
            label_lookup = dict(zip(label_df['BlockId'], label_df['Label']))
        except Exception as e:
            print(f"Error reading label file: {e}")
            return None

        parsed_logs = []
        print(f"Processing {logfile_path}...")
        start_time = time.time()

        with open(logfile_path, 'r', encoding='latin-1') as f:
            for line in f:
                line = line.strip()
                if not line: continue

                # Extract Session ID
                match = block_id_regex.search(line)
                session_id = match.group(1) if match else "_NONE_"

                # Get Label (Default to Normal if not found)
                label = label_lookup.get(session_id, "Normal")

                # Parse log line to get Template ID
                result = parser.add_log_message(line)
                template_id = result["cluster_id"]

                parsed_logs.append({
                    "SessionID": session_id,
                    "TemplateID": template_id,
                    "Label": label
                })

        print(f"Processed in {time.time() - start_time:.2f} seconds.")
        return pd.DataFrame(parsed_logs)

In [ ]:
# [Cell 5] Run the Parser
if files_exist:
    df_combined = parse_hdfs_log(log_file, label_file, parser)

    if df_combined is not None:
        # Save structured data to CSV (optional backup)
        df_combined.to_csv("HDFS_structured.csv", index=False)
        print("Parsing complete. Data saved to HDFS_structured.csv")
        print(f"Total Log Lines: {len(df_combined)}")
        print(f"Unique Templates Found: {len(parser.drain.clusters)}")

In [ ]:
# [Cell 6] Load Data & Create Vocabulary
if files_exist and df_combined is not None:
    # Filter out logs that didn't have a session ID
    df = df_combined[df_combined['SessionID'] != '_NONE_'].copy()

    # Create Mapping: TemplateID -> Integer
    unique_templates = df['TemplateID'].unique()
    template_to_int = {t: i + 1 for i, t in enumerate(unique_templates)}
    template_to_int["<PAD>"] = 0 # Padding (reserved)

    n_templates = len(unique_templates) + 1
    print(f"Vocabulary Size (including padding): {n_templates}")

In [ ]:
# [Cell 7] Generate Sliding Windows
# CRITICAL: We preserve 'SessionID' here for block-level evaluation later.

if files_exist:
    print("Generating sliding windows (Size 10)...")
    window_size = 10
    all_sequences = []

    # Group by SessionID to keep context
    for session_id, group in df.groupby('SessionID'):
        # Convert templates to integers
        temp_seq = [template_to_int[t] for t in group['TemplateID'].tolist()]
        label = group['Label'].iloc[0]

        # Create sliding windows
        if len(temp_seq) > window_size:
            for i in range(len(temp_seq) - window_size):
                all_sequences.append({
                    "SessionID": session_id,
                    "window": temp_seq[i : i + window_size],
                    "next_event": temp_seq[i + window_size],
                    "label": label
                })

    df_sequences = pd.DataFrame(all_sequences)
    print(f"Generated {len(df_sequences)} sequences.")

In [ ]:
# [Cell 8] Split Data (Unsupervised) with Validation Set
from sklearn.model_selection import train_test_split

if 'df_sequences' in locals():
    # 1. Isolate Normal and Abnormal data
    df_normal = df_sequences[df_sequences['label'] == 'Normal']
    df_abnormal = df_sequences[df_sequences['label'] == 'Anomaly']

    # 2. Split Normal data into Train (80%) and Temp (20%)
    df_normal_train_full, df_normal_temp = train_test_split(df_normal, test_size=0.0775, random_state=42)

    # 3. Split Train further into Train (85%) and Validation (15%)
    # This results in roughly: 68% Train, 12% Val, 20% Test (of the original normal data)
    df_train, df_val = train_test_split(df_normal_train_full, test_size=0.0875, random_state=42)

    # 4. Create Final Test Set: Remaining Normal data + ALL Abnormal data
    df_test = pd.concat([df_normal_temp, df_abnormal], ignore_index=True)

    print(f"Training Samples (Normal): {len(df_train)}")
    print(f"Validation Samples (Normal): {len(df_val)}")
    print(f"Testing Samples (Mixed): {len(df_test)}")

In [ ]:
# [Cell 9] Create PyTorch DataLoaders
import torch
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

if not df_train.empty:
    batch_size = 128

    # --- Train Loader ---
    X_train = torch.tensor(np.array(df_train['window'].tolist()), dtype=torch.long)
    y_train = torch.tensor(df_train['next_event'].tolist(), dtype=torch.long)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

    # --- Validation Loader ---
    X_val = torch.tensor(np.array(df_val['window'].tolist()), dtype=torch.long)
    y_val = torch.tensor(df_val['next_event'].tolist(), dtype=torch.long)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    # --- Test Loader (Custom Class) ---
    class TestDataset(torch.utils.data.Dataset):
        def __init__(self, windows, next_events, session_ids):
            self.windows = windows
            self.next_events = next_events
            self.session_ids = session_ids
        def __len__(self): return len(self.windows)
        def __getitem__(self, idx):
            return self.windows[idx], self.next_events[idx], self.session_ids[idx]

    X_test = torch.tensor(np.array(df_test['window'].tolist()), dtype=torch.long)
    y_test = torch.tensor(df_test['next_event'].tolist(), dtype=torch.long)
    test_ids = df_test['SessionID'].values

    test_loader = DataLoader(TestDataset(X_test, y_test, test_ids), batch_size=batch_size, shuffle=False)
    print("DataLoaders (Train, Val, Test) created successfully.")

In [ ]:
# [Cell 10] Define Model Architecture
import torch.nn as nn

class DeepLog(nn.Module):
    def __init__(self, n_templates, embedding_dim, hidden_size, num_layers):
        super(DeepLog, self).__init__()
        self.embedding = nn.Embedding(n_templates, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2  # <--- Dropout added to prevent overfitting
        )
        self.fc = nn.Linear(hidden_size, n_templates)

    def forward(self, x):
        embedded_x = self.embedding(x)
        lstm_out, _ = self.lstm(embedded_x)
        # Take the output of the last time step
        out = self.fc(lstm_out[:, -1, :])
        return out

print("DeepLog Model defined.")

In [ ]:
# [Cell 11] Train the Model with Validation & Best Model Saving
import torch.optim as optim
import torch.nn as nn # Ensure nn is imported if not already in context
from tqdm.notebook import tqdm
import copy

if not df_train.empty:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    # Initialize Model
    # Note: n_templates must be defined from Cell 6
    model = DeepLog(n_templates, 256, 256, 2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Training Configuration
    num_epochs = 50  # UPDATED to 30
    best_acc = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())

    for epoch in range(num_epochs):
        # --- Training Phase ---
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        # Progress bar for training
        progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)

        for windows, next_events in progress:
            windows, next_events = windows.to(device), next_events.to(device)

            optimizer.zero_grad()
            outputs = model(windows)
            loss = criterion(outputs, next_events)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * windows.size(0)
            
            # Track training accuracy
            _, predicted = torch.max(outputs.data, 1)
            train_total += next_events.size(0)
            train_correct += (predicted == next_events).sum().item()

        avg_train_loss = train_loss / len(train_loader.dataset)
        avg_train_acc = train_correct / train_total

        # --- Validation Phase ---
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for windows, next_events in val_loader:
                windows, next_events = windows.to(device), next_events.to(device)
                outputs = model(windows)
                loss = criterion(outputs, next_events)
                
                val_loss += loss.item() * windows.size(0)
                
                _, predicted = torch.max(outputs.data, 1)
                val_total += next_events.size(0)
                val_correct += (predicted == next_events).sum().item()

        avg_val_loss = val_loss / len(val_loader.dataset)
        avg_val_acc = val_correct / val_total

        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f} Acc: {avg_train_acc:.4f} | Val Loss: {avg_val_loss:.4f} Acc: {avg_val_acc:.4f}")

        # --- Save Best Model ---
        if avg_val_acc > best_acc:
            best_acc = avg_val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), "best_model.pth")
            print(f"  --> New Best Validation Accuracy! Model Saved.")

    print(f"\nTraining finished. Best Validation Accuracy: {best_acc:.4f}")
    
    # Load best model weights for final evaluation
    model.load_state_dict(best_model_wts)
    print("Loaded best model weights for evaluation.")

In [ ]:
# [Cell 12] Block-Level Evaluation (Session-based)
from sklearn.metrics import confusion_matrix, classification_report

if not df_train.empty:
    print("Starting Block-Level Evaluation...")
    g = 4  # Top-k threshold (DeepLog standard is often 9 or 10)
    model.eval()

    session_preds = {}
    # Create lookup map for True Labels: SessionID -> Label
    unique_sessions = df_test[['SessionID', 'label']].drop_duplicates()
    true_labels_map = dict(zip(unique_sessions['SessionID'], unique_sessions['label']))

    with torch.no_grad():
        for windows, next_events, batch_ids in tqdm(test_loader, desc="Evaluating"):
            windows, next_events = windows.to(device), next_events.to(device)
            outputs = model(windows)

            # Get indices of top-g probabilities
            _, top_k = torch.topk(outputs, k=g, dim=1)
            top_k = top_k.cpu().numpy()
            true_events = next_events.cpu().numpy()

            for i in range(len(batch_ids)):
                sid = batch_ids[i]
                # Check if the actual next event was in the top k predictions
                # If NOT in top k -> It's an Anomaly (1)
                is_anomaly = 1 if true_events[i] not in top_k[i] else 0

                # Logic: If ANY window in a session is an anomaly, the WHOLE session is anomalous
                if sid not in session_preds:
                    session_preds[sid] = is_anomaly
                else:
                    session_preds[sid] = max(session_preds[sid], is_anomaly)

    # Prepare final lists for metrics
    y_true = []
    y_pred = []

    for sid, pred in session_preds.items():
        if sid in true_labels_map:
            y_pred.append(pred)
            # Convert label string to int (Anomaly=1, Normal=0)
            y_true.append(1 if true_labels_map[sid] == 'Anomaly' else 0)

    # --- Print Results ---
    print("\n--- Final Classification Report ---")
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Anomaly'], digits=4))

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    print("\n--- Confusion Matrix ---")
    print(f"True Negatives (Normal caught as Normal): {tn}")
    print(f"False Positives (Normal flagged as Anomaly): {fp}")
    print(f"False Negatives (Anomaly missed): {fn}")
    print(f"True Positives (Anomaly caught): {tp}")

    if (tp+fn) > 0:
        print(f"\nRecall (Sensitivity): {tp / (tp+fn):.4f}")
    if (tp+fp) > 0:
        print(f"Precision: {tp / (tp+fp):.4f}")

In [ ]:
# [Cell 12] Block-Level Evaluation (Sweep all values of g)
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

if not df_train.empty:
    print("--- 1. Computing Ranks (Inference) ---")
    model.eval()

    # Dictionary to store the worst (highest) rank observed in each session
    # Logic: If the worst rank in a session is >= g, then that session has at least one anomaly.
    session_max_ranks = {}
    
    # Ground truth lookup
    unique_sessions = df_test[['SessionID', 'label']].drop_duplicates()
    true_labels_map = dict(zip(unique_sessions['SessionID'], unique_sessions['label']))

    with torch.no_grad():
        for windows, next_events, batch_ids in tqdm(test_loader, desc="Calculating Ranks"):
            windows, next_events = windows.to(device), next_events.to(device)
            outputs = model(windows)
            
            # Get the sorted order of predictions (indices)
            # We want to know where the 'true' next_event falls in this list
            sorted_indices = torch.argsort(outputs, dim=1, descending=True)
            
            # Find the rank of the true event
            # .unsqueeze(1) makes next_events shape (batch, 1) to broadcast against sorted_indices (batch, n_classes)
            # matches is a boolean tensor where True indicates the position of the target
            matches = (sorted_indices == next_events.unsqueeze(1))
            
            # .nonzero() returns indices [row, col]. The 'col' part is the rank (0-indexed)
            _, ranks = matches.nonzero(as_tuple=True)
            ranks = ranks.cpu().numpy()
            
            for i, sid in enumerate(batch_ids):
                # We only care about the MAX rank in a session 
                # (because if rank >= g for ANY window, the session is Anomaly)
                current_rank = ranks[i]
                if sid not in session_max_ranks:
                    session_max_ranks[sid] = current_rank
                else:
                    session_max_ranks[sid] = max(session_max_ranks[sid], current_rank)

    print("\n--- 2. Sweeping all values of g ---")
    
    # Determine max possible g (vocabulary size)
    max_g = model.fc.out_features
    results = []

    # Prepare Ground Truth vectors (aligned with session_max_ranks keys)
    # We only evaluate sessions that actually had windows in the test set
    eval_sids = list(session_max_ranks.keys())
    y_true = [1 if true_labels_map[sid] == 'Anomaly' else 0 for sid in eval_sids]
    
    # Sweep g
    for g in range(1, max_g + 1):
        # Predict: Anomaly (1) if the worst rank in session >= g (meaning it wasn't in top-g)
        # Normal (0) otherwise
        y_pred = [1 if session_max_ranks[sid] >= g else 0 for sid in eval_sids]
        
        # Calculate metrics (Macro Average or specific to Anomaly class)
        # We focus on the 'Anomaly' class (1) F1-score usually
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
        
        results.append({
            'g': g,
            'Precision': report['1']['precision'],
            'Recall': report['1']['recall'],
            'F1-Score': report['1']['f1-score'],
            'Accuracy': report['accuracy']
        })

    # --- 3. Display Results ---
    res_df = pd.DataFrame(results)
    
    # Sort by F1-Score to find best g
    print("\nTop 5 values of g (by F1-Score):")
    print(res_df.sort_values(by='F1-Score', ascending=False).head(5))

    # Save to CSV
    res_df.to_csv("g_sweep_results.csv", index=False)
    print("\nFull results saved to 'g_sweep_results.csv'")

    # --- 4. Plot F1 Score vs g ---
    plt.figure(figsize=(10, 6))
    plt.plot(res_df['g'], res_df['F1-Score'], marker='o', label='F1-Score (Anomaly)')
    plt.plot(res_df['g'], res_df['Recall'], linestyle='--', label='Recall')
    plt.plot(res_df['g'], res_df['Precision'], linestyle=':', label='Precision')
    
    plt.title(f"Performance vs. Top-g Threshold (Max g={max_g})")
    plt.xlabel("g (Top-k Candidates)")
    plt.ylabel("Score")
    plt.legend()
    plt.grid(True)
    plt.savefig("f1_score_vs_g.png")
    print("Plot saved as 'f1_score_vs_g.png'")
    plt.show()

In [ ]:
import pickle
import json
import torch
import os

working_dir = "/kaggle/working/"

# 1. Save the PyTorch Model
torch.save(model.state_dict(), os.path.join(working_dir, "deeplog_model.pth"))

# 2. Save the Drain3 Parser State
with open(os.path.join(working_dir, "drain3_state.bin"), "wb") as f:
    pickle.dump(parser, f)

# 3. Save the Template-to-Integer Mapping (THIS IS THE FIX)
# We force the keys to be regular strings and values to be regular integers
clean_template_map = {str(k): int(v) for k, v in template_to_int.items()}

with open(os.path.join(working_dir, "template_map.json"), "w") as f:
    json.dump(clean_template_map, f)

print("Artifacts saved successfully to /kaggle/working/!")